In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.tree import DecisionTreeClassifier 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import * 
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv(r"E:\ml_project_classification\notebooks\wine_fraud.csv")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,Legit,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,Legit,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,Legit,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,Legit,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,Legit,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,Legit,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,Legit,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,Legit,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,Legit,white


In [4]:
df_copy = df.copy()
df_copy.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,Legit,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,Legit,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,Legit,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,Legit,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,Legit,red


In [12]:
x=df_copy.drop(['type'],axis=1)
y=df_copy['type']
y=y.map({'red':0,'white':1})
y.unique()

array([0, 1], dtype=int64)

In [13]:
numeric_col = [col for col in x.columns if x[col].dtype!='O']
cat_col = [col for col in x.columns if x[col].dtype=='O']
print("Numeric Column",numeric_col)
print('\n','*' * 50)
print("Catgorical column",cat_col) 

Numeric Column ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

 **************************************************
Catgorical column ['quality']


In [14]:
encoder=OneHotEncoder()
scale =StandardScaler() 
preprocess = ColumnTransformer(
    [
        ('encoding',encoder,cat_col),
        ('scale',scale,numeric_col)
    ]
)
preprocess

ColumnTransformer(transformers=[('encoding', OneHotEncoder(), ['quality']),
                                ('scale', StandardScaler(),
                                 ['fixed acidity', 'volatile acidity',
                                  'citric acid', 'residual sugar', 'chlorides',
                                  'free sulfur dioxide', 'total sulfur dioxide',
                                  'density', 'pH', 'sulphates', 'alcohol'])])

In [17]:
x_scale = preprocess.fit_transform(x)
print('x_scale shape : ',x_scale.shape)
x_scale

x_scale shape :  (6497, 13)


array([[ 0.        ,  1.        ,  0.14247327, ...,  1.81308951,
         0.19309677, -0.91546416],
       [ 0.        ,  1.        ,  0.45103572, ..., -0.11507303,
         0.99957862, -0.58006813],
       [ 0.        ,  1.        ,  0.45103572, ...,  0.25811972,
         0.79795816, -0.58006813],
       ...,
       [ 0.        ,  1.        , -0.55179227, ..., -1.42124765,
        -0.47897144, -0.91546416],
       [ 0.        ,  1.        , -1.32319841, ...,  0.75571005,
        -1.016626  ,  1.9354021 ],
       [ 0.        ,  1.        , -0.93749534, ...,  0.25811972,
        -1.41986693,  1.09691202]])

In [18]:
from sklearn.model_selection import train_test_split 
x_train,x_test,y_train,y_test = train_test_split(x_scale,y,test_size=0.2,random_state=42) 

In [19]:
def evalute_matrics(actual,predict):
    recall = recall_score(actual,predict)
    prec = precision_score(actual,predict)
    f1_sc =f1_score(actual,predict) 
    return recall, prec, f1_sc 


In [22]:
models={
    'logistic regresser': LogisticRegression(),
    'random forest' : RandomForestClassifier(), 
    'decision tree': DecisionTreeClassifier(),
    'Neghbiour' : KNeighborsClassifier()
}


In [24]:
model_list=[]
f1_score_list =[]
for i in range(len(list(models))):
    model=list(models.values())[i] 
    model.fit(x_train,y_train) 

    y_train_pred = model.predict(x_train) 
    y_test_pred = model.predict(x_test) 

    model_train_recall,model_train_prec,model_train_f1_sc = evalute_matrics(y_train,y_train_pred)
    model_test_recall, model_test_prec, model_test_f1_sc = evalute_matrics(y_test,y_test_pred) 

    print(list(models.keys())[i])

    model_list.append(list(models.keys())[i])

    print("Model Perfomance for training data..")
    print("Recall for traing data :",model_train_recall)
    print("Precision for Traing data :",model_train_prec) 
    print("F1 Score for Model Traing data :",model_train_f1_sc)

    print('*'* 50)

    print("Model Perfomance for testing data..")
    print("Recall for testing data :",model_test_recall)
    print("Precision for testing data :",model_test_prec) 
    print("F1 Score for Model testing data :",model_test_f1_sc)

    f1_score_list.append(model_test_f1_sc)

    print('*' * 50 ) 
    print('\n') 






logistic regresser
Model Perfomance for training data..
Recall for traing data : 0.9966996699669967
Precision for Traing data : 0.9956885620086229
F1 Score for Model Traing data : 0.9961938594265415
**************************************************
Model Perfomance for testing data..
Recall for testing data : 0.9947862356621481
Precision for testing data : 0.9906542056074766
F1 Score for Model testing data : 0.9927159209157128
**************************************************


random forest
Model Perfomance for training data..
Recall for traing data : 0.9997461284589998
Precision for Traing data : 0.9997461284589998
F1 Score for Model Traing data : 0.9997461284589998
**************************************************
Model Perfomance for testing data..
Recall for testing data : 0.9989572471324296
Precision for testing data : 0.9937759336099585
F1 Score for Model testing data : 0.9963598543941757
**************************************************


decision tree
Model Perfomance for 

In [ ]:
df_matrics = pd.DataFrame(zip(model_list,f1_score_list),columns=['Model name','F1Score'])
df_matrics # Random Forest working better.. 

,Model name,F1Score
0,logistic regresser,0.992716
1,random forest,0.996360
2,decision tree,0.987474
3,Neghbiour,0.992183
